### preprocess_endmember_gui_single_xml

Generalized GUI tool to:
  1) Read a single XML (specified once) to get valid wavelengths
  2) For each excitation (488 / 561 / 647 / 730): crop reference CSVs to wl_min..wl_max
  3) Interpolate cropped spectra to the XML's valid wavelengths

Changes from previous version:
- XML file is specified ONCE (global), used for all channels
- Each channel still has editable wl_min/max and can add multiple CSVs

Dependencies: pandas, numpy, tkinter

In [3]:
# -*- coding: utf-8 -*-
import os
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
import pandas as pd
import numpy as np
from xml.etree import ElementTree as ET
from typing import List, Dict

# ---------------------- Core functions ----------------------

def get_valid_wavelengths(xml_path: str, wl_min: float, wl_max: float) -> List[float]:
    """Parse XML and collect band wavelengths within [wl_min, wl_max]."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    wavelengths: List[float] = []
    for band in root.findall(".//band"):
        try:
            wl = float(band.attrib.get("wavelength_nm", "0"))
        except ValueError:
            continue
        if wl_min <= wl <= wl_max and wl != 0:
            wavelengths.append(wl)
    wavelengths.sort()
    return wavelengths

def crop_reference_csv(in_path: str, wl_min: float, wl_max: float, out_path: str) -> str | None:
    """Read CSV, drop first 4 columns, keep numeric columns within [wl_min, wl_max]."""
    df = pd.read_csv(in_path)
    if df.shape[1] <= 4:
        print(f"⚠️ {os.path.basename(in_path)} : not enough columns (<=4) – skipped")
        return None

    data = df.iloc[:, 4:]  # drop first 4 metadata columns

    def _is_numeric_col(colname: str) -> bool:
        try:
            float(colname)
            return True
        except Exception:
            return False

    valid_cols = [c for c in data.columns if _is_numeric_col(c) and wl_min <= float(c) <= wl_max]
    if not valid_cols:
        print(f"⚠️ {os.path.basename(in_path)} : no wavelengths in range – skipped")
        return None

    cropped = data[valid_cols]
    cropped.to_csv(out_path, index=False)
    print(f"  ✅ Cropped → {out_path}")
    return out_path

def interpolate_to_xml_wavelengths(crop_path: str, xml_wavelengths: List[float], out_path: str):
    """Interpolate each row of cropped CSV to the wavelengths defined by the XML."""
    df = pd.read_csv(crop_path)
    x = np.array([float(c) for c in df.columns], dtype=float)
    out_records = []
    for _, row in df.iterrows():
        y = row.to_numpy(dtype=float)
        y_interp = np.interp(xml_wavelengths, x, y)
        out_records.append(y_interp)

    out_df = pd.DataFrame(out_records, columns=[f"{w:.2f}" for w in xml_wavelengths])
    out_df.to_csv(out_path, index=False)
    print(f"  ✅ Interpolated → {out_path}")

# ---------------------- GUI application ----------------------

DEFAULTS = {
    488: dict(wl_min=500.0, wl_max=700.0),
    561: dict(wl_min=575.0, wl_max=840.0),
    647: dict(wl_min=675.0, wl_max=900.0),
    730: dict(wl_min=750.0, wl_max=900.0),
}

class ChannelUI(ttk.LabelFrame):
    """Per-excitation UI block with: wl_min/max, CSV list (XML is global)."""
    def __init__(self, master, wavelength: int):
        super().__init__(master, text=f"ex{wavelength}")
        self.wl = wavelength

        # Variables
        self.var_min = tk.DoubleVar(value=DEFAULTS[wavelength]["wl_min"])
        self.var_max = tk.DoubleVar(value=DEFAULTS[wavelength]["wl_max"])

        # Selected CSV files list
        self.csv_files: List[str] = []

        # Layout
        self._build()

    def _build(self):
        pad = {"padx": 6, "pady": 3}

        # wl_min / wl_max
        row = 0
        ttk.Label(self, text="wl_min").grid(row=row, column=0, sticky="e", **pad)
        ttk.Entry(self, textvariable=self.var_min, width=10).grid(row=row, column=1, sticky="w", **pad)
        ttk.Label(self, text="wl_max").grid(row=row, column=2, sticky="e", **pad)
        ttk.Entry(self, textvariable=self.var_max, width=10).grid(row=row, column=3, sticky="w", **pad)

        # CSV file list and buttons
        row += 1
        self.listbox = tk.Listbox(self, height=5, width=70)
        self.listbox.grid(row=row, column=0, columnspan=4, sticky="we", **pad)

        row += 1
        btns = ttk.Frame(self)
        btns.grid(row=row, column=0, columnspan=4, sticky="w", **pad)
        ttk.Button(btns, text="Add CSVs…", command=self._add_csvs).grid(row=0, column=0, padx=4)
        ttk.Button(btns, text="Remove selected", command=self._remove_selected).grid(row=0, column=1, padx=4)
        ttk.Button(btns, text="Clear list", command=self._clear).grid(row=0, column=2, padx=4)

    def _add_csvs(self):
        files = filedialog.askopenfilenames(
            title=f"Add CSV reference files (ex{self.wl})",
            filetypes=[("CSV files", "*.csv"), ("All files", "*.*")],
        )
        if files:
            for f in files:
                if f not in self.csv_files:
                    self.csv_files.append(f)
                    self.listbox.insert(tk.END, f)

    def _remove_selected(self):
        sel = list(self.listbox.curselection())
        sel.reverse()
        for idx in sel:
            path = self.listbox.get(idx)
            if path in self.csv_files:
                self.csv_files.remove(path)
            self.listbox.delete(idx)

    def _clear(self):
        self.csv_files.clear()
        self.listbox.delete(0, tk.END)

    def params(self) -> Dict:
        return dict(
            wl=self.wl,
            wl_min=float(self.var_min.get()),
            wl_max=float(self.var_max.get()),
            csvs=list(self.csv_files),
        )

class App(ttk.Frame):
    def __init__(self, master):
        super().__init__(master)
        master.title("Endmember CSV Crop & Interpolate — single XML for all (488 / 561 / 647 / 730)")
        master.protocol("WM_DELETE_WINDOW", master.destroy)
        self.grid(padx=10, pady=10, sticky="nsew")

        # Output root + Global XML
        self.var_out = tk.StringVar(value=os.path.join(os.path.expanduser("~"), "endmember_preprocessed"))
        self.var_xml = tk.StringVar(value="")
        self._build_header()

        # Four channels
        self.channels = {}
        row = 1
        for wl in (488, 561, 647, 730):
            ch = ChannelUI(self, wl)
            ch.grid(row=row, column=0, sticky="nsew", pady=(6, 2))
            self.channels[wl] = ch
            row += 1

        # Run button
        runf = ttk.Frame(self)
        runf.grid(row=row, column=0, sticky="e", pady=(8, 0))
        ttk.Button(runf, text="Run", command=self.on_run).grid(row=0, column=0, padx=6)
        ttk.Button(runf, text="Quit", command=self.master.destroy).grid(row=0, column=1, padx=6)

        # Status
        self.status = tk.StringVar(value="Ready.")
        ttk.Label(self, textvariable=self.status).grid(row=row+1, column=0, sticky="w", pady=(6, 0))

    def _build_header(self):
        f = ttk.LabelFrame(self, text="Global Settings")
        f.grid(row=0, column=0, sticky="ew")
        # Output root
        ttk.Label(f, text="Output root").grid(row=0, column=0, padx=6, pady=6, sticky="e")
        ttk.Entry(f, textvariable=self.var_out, width=60).grid(row=0, column=1, padx=6, pady=6)
        ttk.Button(f, text="Browse", command=self._choose_out).grid(row=0, column=2, padx=6, pady=6)
        # Single XML
        ttk.Label(f, text="XML file (used for all channels)").grid(row=1, column=0, padx=6, pady=6, sticky="e")
        ttk.Entry(f, textvariable=self.var_xml, width=60).grid(row=1, column=1, padx=6, pady=6)
        ttk.Button(f, text="Browse", command=self._choose_xml).grid(row=1, column=2, padx=6, pady=6)

    def _choose_out(self):
        d = filedialog.askdirectory(title="Choose output root")
        if d:
            self.var_out.set(d)

    def _choose_xml(self):
        f = filedialog.askopenfilename(title="Choose XML",
                                       filetypes=[("XML files", "*.xml"), ("All files", "*.*")])
        if f:
            self.var_xml.set(f)

    def on_run(self):
        out_root = self.var_out.get().strip()
        xml_path = self.var_xml.get().strip()

        if not out_root:
            messagebox.showerror("Output", "Please choose an output root.")
            return
        if not xml_path:
            messagebox.showerror("XML", "Please choose the XML file (used for all channels).")
            return

        # Check that we have at least one CSV
        any_files = any(ch.csv_files for ch in self.channels.values())
        if not any_files:
            messagebox.showwarning("No input", "No CSVs provided. Add CSV files for at least one excitation.")
            return

        # Process each channel
        try:
            for wl, ch in self.channels.items():
                p = ch.params()
                csvs   = p["csvs"]
                if not csvs:
                    continue

                wl_min = p["wl_min"]
                wl_max = p["wl_max"]

                # Create subfolder
                wl_out = os.path.join(out_root, f"ex{wl}")
                os.makedirs(wl_out, exist_ok=True)

                # Parse XML wavelengths for this channel's range
                try:
                    xml_wavelengths = get_valid_wavelengths(xml_path, wl_min, wl_max)
                except Exception as e:
                    messagebox.showerror(f"ex{wl} XML", f"Failed to parse XML: {e}")
                    return
                if not xml_wavelengths:
                    messagebox.showwarning(f"ex{wl}", "No valid wavelengths in the specified range. Skipped.")
                    continue

                # Process CSVs
                for in_csv in csvs:
                    fname = os.path.basename(in_csv)
                    crop_out   = os.path.join(wl_out, f"Crop_{fname}")
                    interp_out = os.path.join(wl_out, f"Interp_crop_{fname}")

                    # Crop
                    cropped_path = crop_reference_csv(in_csv, wl_min, wl_max, crop_out)
                    if not cropped_path:
                        continue
                    # Interpolate
                    try:
                        interpolate_to_xml_wavelengths(cropped_path, xml_wavelengths, interp_out)
                    except Exception as e:
                        print(f"❌ Interp failed for {fname}: {e}")

            self.status.set("Done. See output folders for results.")
            messagebox.showinfo("Finished", "All selected CSVs have been processed.")

        except Exception as e:
            self.status.set("Error occurred.")
            messagebox.showerror("Error", str(e))

def main():
    root = tk.Tk()
    try:
        root.call("source", "azure.tcl")
        root.call("set_theme", "light")
    except Exception:
        pass
    App(root)
    root.mainloop()

if __name__ == "__main__":
    main()


### extract_valid_bands_gui
GUI tool to extract wavelength band ranges from hyperspectral TIFF stacks
for four excitations (ex488 / ex561 / ex647 / ex730).

Key features:
- Single (global) XML can be specified and shared across all channels.
  If not provided, the XML alongside the selected Loop1 TIFF for each channel is used.
- Per-channel editable wavelength ranges (wl_min / wl_max) with CSV-version defaults:
    ex488: 500–700 nm
    ex561: 575–840 nm
    ex647: 675–900 nm
    ex730: 750–900 nm
- For each channel, select ONE TIFF (e.g., loop1_***_object_300ms.tiff);
  the app will batch-process all matching loop* files in the same folder.
- Output: <Output root>/Extracted_ex{wl}/Extracted_<original_name>.tif(f)

Dependencies: tkinter, numpy, tifffile, xml.etree.ElementTree

In [4]:
# -*- coding: utf-8 -*-
import os
import re
import tkinter as tk
from tkinter import ttk, filedialog, messagebox
from pathlib import Path
from typing import List, Dict, Optional

import numpy as np
import tifffile as tiff
from xml.etree import ElementTree as ET

# ---------------------- Core helpers ----------------------

def get_valid_band_indices(xml_path: Path, wl_min: float, wl_max: float) -> List[int]:
    """Parse bands from XML and return indices whose wavelength_nm is in [wl_min, wl_max]."""
    root = ET.parse(xml_path).getroot()
    idxs: List[int] = []
    for idx, band in enumerate(root.findall(".//band")):
        try:
            wl = float(band.attrib.get("wavelength_nm", "0"))
        except ValueError:
            wl = 0.0
        if wl_min <= wl <= wl_max and wl != 0:
            idxs.append(idx)
    return idxs

def extract_tiff_bands(tiff_path: Path, indices: List[int], out_path: Path):
    """Read multi-page/multi-slice TIFF and save only selected bands (indices)."""
    # Read entire stack (Z,Y,X)
    arr = tiff.imread(str(tiff_path))
    if arr.ndim == 2:
        # Single plane: nothing to extract meaningfully; skip or save as-is
        sel = arr[None, ...]
    else:
        sel = arr[indices, ...]
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tiff.imwrite(str(out_path), sel, photometric="minisblack")
    print(f"  ✓ {out_path.name}")

# ---------------------- GUI ----------------------

DEFAULT_RANGES = {
    488: dict(wl_min=500.0, wl_max=700.0),
    561: dict(wl_min=575.0, wl_max=840.0),
    647: dict(wl_min=675.0, wl_max=900.0),
    730: dict(wl_min=750.0, wl_max=900.0),
}

LOOP_FILE_REGEX = re.compile(
    r"^loop\d+_(?P<wl>\d{3})nm_object_(?P<exp>\d+ms)\.(tif|tiff)$",
    re.IGNORECASE
)

class ChannelUI(ttk.LabelFrame):
    """Per-excitation input: one TIFF (Loop1 or any loop), wl_min/max."""
    def __init__(self, master, wavelength: int):
        super().__init__(master, text=f"ex{wavelength}")
        self.wl = wavelength
        self.var_tiff = tk.StringVar(value="")
        self.var_min  = tk.DoubleVar(value=DEFAULT_RANGES[wavelength]["wl_min"])
        self.var_max  = tk.DoubleVar(value=DEFAULT_RANGES[wavelength]["wl_max"])
        self._build()

    def _build(self):
        pad = {"padx": 6, "pady": 3}

        ttk.Label(self, text="Loop1 TIFF (or any loop)").grid(row=0, column=0, sticky="e", **pad)
        ttk.Entry(self, textvariable=self.var_tiff, width=58).grid(row=0, column=1, sticky="we", **pad)
        ttk.Button(self, text="Browse", command=self._choose_tiff).grid(row=0, column=2, **pad)

        ttk.Label(self, text="wl_min").grid(row=1, column=0, sticky="e", **pad)
        ttk.Entry(self, textvariable=self.var_min, width=10).grid(row=1, column=1, sticky="w", **pad)
        ttk.Label(self, text="wl_max").grid(row=1, column=2, sticky="e", **pad)
        ttk.Entry(self, textvariable=self.var_max, width=10).grid(row=1, column=3, sticky="w", **pad)

    def _choose_tiff(self):
        f = filedialog.askopenfilename(
            title=f"Choose a TIFF for ex{self.wl}",
            filetypes=[("TIFF files", "*.tif *.tiff"), ("All files", "*.*")]
        )
        if f:
            self.var_tiff.set(f)

    def params(self) -> Dict:
        return dict(
            wl=self.wl,
            tiff_path=self.var_tiff.get().strip(),
            wl_min=float(self.var_min.get()),
            wl_max=float(self.var_max.get()),
        )

class App(ttk.Frame):
    def __init__(self, master):
        super().__init__(master)
        master.title("Extract valid bands from hyperspectral TIFFs (ex488/ex561/ex647/ex730)")
        master.protocol("WM_DELETE_WINDOW", master.destroy)
        self.grid(padx=10, pady=10, sticky="nsew")

        # Global settings (Output root + optional single XML)
        self.var_out = tk.StringVar(value=os.path.join(os.path.expanduser("~"), "Extracted_bands"))
        self.var_xml = tk.StringVar(value="")  # optional shared XML for all channels
        self._build_header()

        # Channel blocks
        self.channels: Dict[int, ChannelUI] = {}
        row = 1
        for wl in (488, 561, 647, 730):
            ch = ChannelUI(self, wl)
            ch.grid(row=row, column=0, sticky="nsew", pady=(6, 2))
            self.channels[wl] = ch
            row += 1

        # Controls
        ctrl = ttk.Frame(self)
        ctrl.grid(row=row, column=0, sticky="e", pady=(8, 0))
        ttk.Button(ctrl, text="Run", command=self.on_run).grid(row=0, column=0, padx=6)
        ttk.Button(ctrl, text="Quit", command=self.master.destroy).grid(row=0, column=1, padx=6)

        # Status
        self.status = tk.StringVar(value="Ready.")
        ttk.Label(self, textvariable=self.status).grid(row=row+1, column=0, sticky="w", pady=(6, 0))

    def _build_header(self):
        f = ttk.LabelFrame(self, text="Global Settings")
        f.grid(row=0, column=0, sticky="ew")
        ttk.Label(f, text="Output root").grid(row=0, column=0, padx=6, pady=6, sticky="e")
        ttk.Entry(f, textvariable=self.var_out, width=60).grid(row=0, column=1, padx=6, pady=6)
        ttk.Button(f, text="Browse", command=self._choose_out).grid(row=0, column=2, padx=6, pady=6)

        ttk.Label(f, text="Single XML for all channels (optional)").grid(row=1, column=0, padx=6, pady=6, sticky="e")
        ttk.Entry(f, textvariable=self.var_xml, width=60).grid(row=1, column=1, padx=6, pady=6)
        ttk.Button(f, text="Browse", command=self._choose_xml).grid(row=1, column=2, padx=6, pady=6)

    def _choose_out(self):
        d = filedialog.askdirectory(title="Choose output root")
        if d:
            self.var_out.set(d)

    def _choose_xml(self):
        f = filedialog.askopenfilename(title="Choose XML",
                                       filetypes=[("XML files", "*.xml"), ("All files", "*.*")])
        if f:
            self.var_xml.set(f)

    def on_run(self):
        out_root = Path(self.var_out.get().strip())
        global_xml = Path(self.var_xml.get().strip()) if self.var_xml.get().strip() else None

        if not out_root:
            messagebox.showerror("Output", "Please choose an output root.")
            return

        # At least one channel should have input
        any_input = any(self.channels[wl].params()["tiff_path"] for wl in self.channels)
        if not any_input:
            messagebox.showwarning("No input", "Please select at least one TIFF (Loop1 or any loop) for a channel.")
            return

        try:
            for wl, ch in self.channels.items():
                p = ch.params()
                tiff_path_str = p["tiff_path"]
                if not tiff_path_str:
                    continue  # no input for this channel

                tiff_path = Path(tiff_path_str)
                if not tiff_path.exists():
                    messagebox.showerror(f"ex{wl}", f"TIFF not found:\n{tiff_path}")
                    return

                wl_min = p["wl_min"]
                wl_max = p["wl_max"]

                # Decide XML: global → per-file (loop1) fallback
                xml_path: Optional[Path] = None
                if global_xml and global_xml.exists():
                    xml_path = global_xml
                else:
                    # Use the sidecar XML next to the selected TIFF
                    candidate = tiff_path.with_suffix(tiff_path.suffix + ".xml")  # e.g., .tiff.xml
                    if candidate.exists():
                        xml_path = candidate
                    else:
                        messagebox.showerror(f"ex{wl}", "XML not found (global not set and sidecar missing).")
                        return

                # Compute valid band indices
                band_idx = get_valid_band_indices(xml_path, wl_min, wl_max)
                if not band_idx:
                    messagebox.showwarning(f"ex{wl}", f"No bands in {wl_min}–{wl_max} nm. Skipped.")
                    continue
                print(f"\n===== ex{wl} =====")
                print(f"  → valid bands: {len(band_idx)} indices from {xml_path.name}")

                # Find all matching loop files in same folder
                in_dir = tiff_path.parent
                # Match patterns like loopX_{wl}nm_object_XXXms.tif(f)
                matched: List[Path] = []
                for f in in_dir.iterdir():
                    if not f.is_file():
                        continue
                    m = LOOP_FILE_REGEX.match(f.name)
                    if m and int(m.group("wl")) == wl:
                        matched.append(f)

                if not matched:
                    messagebox.showwarning(f"ex{wl}", "No matching loop* files found in the folder. Skipped.")
                    continue

                # Output folder
                ch_out = out_root / f"Extracted_ex{wl}"
                ch_out.mkdir(parents=True, exist_ok=True)

                # Process
                for f in sorted(matched, key=lambda p: p.name.lower()):
                    # Preserve original filename, add "Extracted_" prefix
                    out_name = f"Extracted_{f.name}"
                    out_path = ch_out / out_name
                    try:
                        extract_tiff_bands(f, band_idx, out_path)
                    except Exception as e:
                        print(f"❌ Failed: {f.name} -> {e}")

            self.status.set("Done. Check the Extracted_ex*** folders.")
            messagebox.showinfo("Finished", "All requested channels have been processed.")

        except Exception as e:
            self.status.set("Error occurred.")
            messagebox.showerror("Error", str(e))

def main():
    root = tk.Tk()
    try:
        root.call("source", "azure.tcl")
        root.call("set_theme", "light")
    except Exception:
        pass
    App(root)
    root.mainloop()

if __name__ == "__main__":
    main()
